## Step 1 — Check GPU

In [9]:
import subprocess, sys

# Fix numpy conflict once and for all
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy==1.26.4",
                "timm",
                "albumentations",
                "grad-cam",
                "facenet-pytorch"])

# MUST restart after this
print("✓ Done — now go to Runtime → Restart Session")

✓ Done — now go to Runtime → Restart Session


In [1]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name    :', torch.cuda.get_device_name(0))
    print('VRAM        :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

GPU available: True
GPU name    : Tesla T4
VRAM        : 15.6 GB


## Step 2 — Mount Drive & Set Project Path

In [3]:
from google.colab import drive
drive.mount('/content/drive')


PROJECT_DIR = '/content/drive/MyDrive/deepfake-detection'

import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/deepfake-detection
Contents: ['README.md', 'setup.py', '.gitignore', 'requirements.txt', 'config.yaml', 'notebooks', '.vscode', 'logs', 'data', 'checkpoints', 'results', 'src']


## Step 3 — Install Dependencies

In [4]:

!pip install -q timm albumentations grad-cam facenet-pytorch
print('Dependencies installed ✓')

Dependencies installed ✓


## Step 4 — Verify Dataset

In [5]:
import yaml
from src.dataset import load_config, collect_image_paths

cfg = load_config('config.yaml')
d   = cfg['data']

for split, rdir, fdir in [
    ('Train', d['train_real'], d['train_fake']),
    ('Val',   d['val_real'],   d['val_fake']),
    ('Test',  d['test_real'],  d['test_fake']),
]:
    r = collect_image_paths(rdir)
    f = collect_image_paths(fdir)
    print(f'{split:<6} → Real: {len(r):>7,}  Fake: {len(f):>7,}  Total: {len(r)+len(f):>8,}')

print('\nDataset verified ✓')

Train  → Real:  15,000  Fake:  15,000  Total:   30,000
Val    → Real:   7,000  Fake:   7,000  Total:   14,000
Test   → Real:   5,413  Fake:   5,492  Total:   10,905

Dataset verified ✓


In [6]:
#these 2 cells are for faster training
with open('src/dataset.py', 'r') as f:
    code = f.read()

# Cap val and test too
old = "    train_paths, train_labels = load_split(d[\"train_real\"], d[\"train_fake\"], cap,  \"Train\")\n    val_paths,   val_labels   = load_split(d[\"val_real\"],   d[\"val_fake\"],   None, \"Val\")\n    test_paths,  test_labels  = load_split(d[\"test_real\"],  d[\"test_fake\"],  None, \"Test\")"

new = """    val_cap  = d.get("max_val_samples_per_class", 2000)
    train_paths, train_labels = load_split(d["train_real"], d["train_fake"], cap,     "Train")
    val_paths,   val_labels   = load_split(d["val_real"],   d["val_fake"],   val_cap, "Val")
    test_paths,  test_labels  = load_split(d["test_real"],  d["test_fake"],  None,    "Test")"""

code = code.replace(old, new)

with open('src/dataset.py', 'w') as f:
    f.write(code)

print('✓ dataset.py patched — val capped at 2000 per class')

✓ dataset.py patched — val capped at 2000 per class


In [9]:
import yaml

with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['data']['max_samples_per_class'] = 15000  # only change from last run
cfg['data']['max_val_samples_per_class'] = 2000
cfg['data']['image_size'] = 128
cfg['training']['batch_size'] = 128
cfg['training']['epochs'] = 15         # early stopping handles this
cfg['training']['early_stopping_patience'] = 4
cfg['training']['learning_rate'] = 1.0e-4
cfg['model']['freeze_epochs'] = 2

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('✓ Config updated')

✓ Config updated


## Step 5 — Run Training

In [6]:
# This runs the full training pipeline:
# - Freeze backbone 3 epochs → unfreeze → full fine-tune
# - Early stopping (patience=4)
# - Best model saved to checkpoints/best_model.pth
# - Plots saved to results/plots/
# - Test set evaluation at the end

from src.train import train
history, best_auc = train('config.yaml')
print(f'\nTraining complete! Best Val AUC: {best_auc:.4f}')


  Deepfake Detection — Training
  Device : cuda
  GPU    : Tesla T4

[Dataset] Loading pre-split Kaggle dataset...
  Capping training at 15,000 images per class
  Train  → Real: 15,000  Fake: 15,000  Total:  30,000
  Val    → Real:  2,000  Fake:  2,000  Total:   4,000
  Test   → Real:  5,413  Fake:  5,492  Total:  10,905
  Class weights → Real: 1.000 | Fake: 1.000

[Model] efficientnet_b0 loaded
  Total params      : 4,664,957
  Trainable params  : 4,664,957
  Backbone frozen. Training classifier head only.

  Starting training for 15 epochs...
  Freeze schedule: backbone frozen for first 2 epochs

Epoch [  1/15]


  Train → Loss: 7.7001 | Acc: 0.6254 | AUC: 0.6480 | F1: 0.6258
  Val   → Loss: 3.6675 | Acc: 0.7310 | AUC: 0.7778 | F1: 0.7448 | LR: 9.86e-05
  ✓ Best model saved → checkpoints/best_model.pth

Epoch [  2/15]


  Train → Loss: 6.0903 | Acc: 0.6717 | AUC: 0.7041 | F1: 0.6687
  Val   → Loss: 3.1445 | Acc: 0.7400 | AUC: 0.7942 | F1: 0.7551 | LR: 9.43e-05
  ✓ Best model saved → checkpoints/best_model.pth

Epoch [  3/15]
  Backbone unfrozen. Full fine-tuning enabled.


  Train → Loss: 5.3554 | Acc: 0.6935 | AUC: 0.7303 | F1: 0.6923
  Val   → Loss: 3.0478 | Acc: 0.7648 | AUC: 0.8141 | F1: 0.7853 | LR: 9.85e-06
  ✓ Best model saved → checkpoints/best_model.pth

Epoch [  4/15]


  Train → Loss: 4.7004 | Acc: 0.7208 | AUC: 0.7615 | F1: 0.7191
  Val   → Loss: 2.5686 | Acc: 0.7895 | AUC: 0.8440 | F1: 0.8035 | LR: 9.40e-06
  ✓ Best model saved → checkpoints/best_model.pth

Epoch [  5/15]


...

  Train → Loss: 2.3960 | Acc: 0.8067 | AUC: 0.8601 | F1: 0.8050
  Val   → Loss: 1.7462 | Acc: 0.8480 | AUC: 0.8993 | F1: 0.8614 | LR: 1.00e-06

Epoch [ 15/15]


  Train → Loss: 2.3582 | Acc: 0.8051 | AUC: 0.8585 | F1: 0.8034
  Val   → Loss: 1.7898 | Acc: 0.8480 | AUC: 0.9000 | F1: 0.8615 | LR: 1.15e-06
  ✓ Best model saved → checkpoints/best_model.pth


  Training complete in 46.6 min
  Best Val AUC : 0.9000
  Best model   : checkpoints/best_model.pth
  Training curves → results/plots/training_curves.png

  Training curves saved to results/plots/

[Evaluation] Running on test set...



[Model] efficientnet_b0 loaded
  Total params      : 4,664,957
  Trainable params  : 4,664,957
  Loaded checkpoint from epoch 15 (val AUC: 0.9000)


  Evaluating test set: 100%|██████████████████████████████| 86/86 [00:51<00:00,  1.67it/s]



  TEST SET RESULTS
  Accuracy  : 0.7479  (74.79%)
  AUC-ROC   : 0.8244
  EER       : 0.2310  (23.10%)
  F1-Score  : 0.7738
  Precision : 0.7060
  Recall    : 0.8560

  Classification Report:
              precision    recall  f1-score   support

        Real       0.81      0.64      0.72      5413
        Fake       0.71      0.86      0.77      5492

    accuracy                           0.75     10905
   macro avg       0.76      0.75      0.74     10905
weighted avg       0.76      0.75      0.74     10905


  Saving evaluation plots...
  ROC curve saved → results/plots/roc_curve.png
  PR curve saved  → results/plots/pr_curve.png
  Confusion matrix → results/plots/confusion_matrix.png
  Score dist.     → results/plots/score_distribution.png
  Benchmark plot  → results/plots/benchmark_comparison.png

  Metrics saved   → results/metrics/test_metrics.json
  Benchmark CSV   → results/metrics/benchmark_comparison.csv

Training complete! Best Val AUC: 0.9000


## Step 6 — Generate Grad-CAM Visualizations

In [7]:
import torch
from src.dataset import build_dataloaders
from src.model import build_model, load_checkpoint
from src.gradcam import generate_gradcam_grid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = build_model(cfg, device)
load_checkpoint(model, cfg['paths']['best_model'], device=device)

_, _, test_loader, _ = build_dataloaders(cfg, verbose=False)

generate_gradcam_grid(model, test_loader, cfg, device, n_samples=16)
print('Grad-CAM visualizations saved ✓')


[Model] efficientnet_b0 loaded
  Total params      : 4,664,957
  Trainable params  : 4,664,957
  Loaded checkpoint from epoch 15 (val AUC: 0.9000)
  Train  → Real: 15,000  Fake: 15,000  Total:  30,000
  Val    → Real:  2,000  Fake:  2,000  Total:   4,000
  Test   → Real:  5,413  Fake:  5,492  Total:  10,905


  Generating Grad-CAM: 100%|██████████| 16/16 [00:01<00:00, 10.34it/s]



  Grad-CAM grid saved → results/gradcam/gradcam_grid.png
  Individual maps    → results/gradcam/
Grad-CAM visualizations saved ✓


## Step 7 — View Results

In [9]:
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Print final metrics
with open('results/metrics/test_metrics.json') as f:
    m = json.load(f)

print('='*45)
print('  FINAL TEST SET RESULTS')
print('='*45)
print(f"  AUC-ROC   : {m['auc']:.4f}")
print(f"  EER       : {m['eer']:.4f}  ({m['eer']*100:.2f}%)")
print(f"  Accuracy  : {m['accuracy']:.4f}  ({m['accuracy']*100:.2f}%)")
print(f"  F1-Score  : {m['f1']:.4f}")
print(f"  Precision : {m['precision']:.4f}")
print(f"  Recall    : {m['recall']:.4f}")
print('='*45)

  FINAL TEST SET RESULTS
  AUC-ROC   : 0.8244
  EER       : 0.2310  (23.10%)
  Accuracy  : 0.7479  (74.79%)
  F1-Score  : 0.7738
  Precision : 0.7060
  Recall    : 0.8560


In [ ]:
# Display all result plots inline
plots = [
    ('results/plots/training_curves.png',    'Training Curves'),
    ('results/plots/roc_curve.png',          'ROC Curve'),
    ('results/plots/confusion_matrix.png',   'Confusion Matrix'),
    ('results/plots/benchmark_comparison.png','Benchmark Comparison'),
    ('results/plots/score_distribution.png', 'Score Distribution'),
    ('results/gradcam/gradcam_grid.png',     'Grad-CAM Visualizations'),
]

for path, title in plots:
    if os.path.exists(path):
        fig, ax = plt.subplots(figsize=(12, 7))
        ax.imshow(mpimg.imread(path))
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'Not found: {path}')

## Step 8 — Download Model to Local Machine

Since the project is in Google Drive, your trained model is already saved at:
```
deepfake-detection/checkpoints/best_model.pth
```
Just download it from Drive to your local `checkpoints/` folder.

In [8]:
# Optional: also download directly from Colab to your browser
from google.colab import files
files.download('checkpoints/best_model.pth')
print('Model download started — check your browser downloads')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model download started — check your browser downloads
